# **Fiesta - An introduction**

This notebook is meant as a tutorial for someone starting completely new with fiesta.

We will first look at installation and setup and then continue to train a machine-learning surrogate for a KN model.

## **1. First steps**

The very first step would be to install fiesta. Since it's a python package, it's best to set up a conda environment and then use the pip install command in that environment.

We assume here that you did set up a suitable conda environment and execute this notebook in this environment. Only then you should execute the next two cells.

##### **a) Plain pip installation**

The easiest way to install fiesta is directly from the pypi database.

That requires an internet connection, as it will try to download all the relevant files from the database and then install them at a suitable location in the environment.

<font color='red'>Warning: Only execute the following cell in an environment where you want to have fiesta installed! </font>

In [ ]:
!python -m pip install fiestaEM[gpu]

##### **b) Installation from source**

Now, the latest version on pypi might not always agree with the latest development version available on github.

Also, if you really want to add something to the code, it's probably best to download the source code from github and then use pip to make an editable install.

This is done with the following commands

<font color='red'>Warning: Only execute the following cell in an environment where you want to have fiesta installed! </font>

In [ ]:
!git clone git@github.com:nuclear-multimessenger-astronomy/fiestaEM.git ./fiestaEM
!python -m pip install -e ./fiestaEM

Now the code is in the new ``fiestaEM`` directory where it can be edited there directly and all the changes will apply once you restart your test program.

##### **c) Downloading machine learning surrogates and data**

Initially, the installation comes with just the source code. 
That means all functions and analytical models can be used immediately.
However, the machine learning surrogates are stored in large pickle files that require an extra download from our huggingface repository.
Luckily, this can easily be done with built-in fiesta utilities.

First we check which types of surrogates are available:

In [ ]:
from fiesta.surrogates import print_built_in_surrogates

print_built_in_surrogates()

Then we can use the ``download_surrogate`` function to download (which here also means immediate installation) a specific model. To see which models are available for download, run

In [ ]:
from fiesta.surrogates import print_downloadable_surrogates

print_downloadable_surrogates()

Now in this case we want to download the ``afgpy_tophat_CVAE`` model. The download is usually only a couple MB.

In [ ]:
from fiesta.surrogates import download_surrogate
success, files = download_surrogate("afgpy_tophat_CVAE")

if success:
    print(f"Download worked! The surrogate .pkl files are now in {files}.")
else:
    print(f"Download did not work :/ Maybe check the internet connection?")

In fact, there is even a function that downloads all the recommended surrogates

In [ ]:
from fiesta.surrogates import download_recommended_surrogates

download_recommended_surrogates()

##### **d) Using a model to predict flux densities and light curves**

Now that we have downloaded and installed all models we need, we can start to use them.

There are two different kinds of models in ``fiesta``. One are machine-learning surrogates, the other simpler analytical models.

The way they are loaded differs. We first load a machine-learning surrogate named ``Bu2026_MLP``.

In [ ]:
from fiesta.inference.lightcurve_model import FluxModel

bu2026_mlp = FluxModel(
    name="Bu2026_MLP", # the name of the surrogate
    filters = ["besselli", "bessellux"], # the photometric filters with which to load it
    # directory = "path_to_your_specific_pkl_files" # which directory to load the files from (If none is given, it will try to find the files in the installation where it was saved by the download utilities).
)


The analytical models are tied to specific classes, which have to be imported and are then just ready to use:

In [ ]:
from fiesta.inference.analytical_models import MetzgerFullModel

metzger2017 = MetzgerFullModel(filters=["besselli", "bessellux"])

Upon loading a surrogate, a lot of useful information is printed, that can also be accessed through the attributes of the model instance:

In [ ]:
print("Bu2026_MLP parameter names: ", bu2026_mlp.parameter_names)
print("Bu2026_MLP parameter ranges: ", bu2026_mlp.parameter_distributions)


print("\n \n")
print("Bu2026_MLP photometric filters: ", bu2026_mlp.filters)
print("Bu2026_MLP photometric time array (days, source-frame): ", bu2026_mlp.times)

##### **Development problem 1:**
*The analytical and surrogate models have no common base class, meaning that changes for one do not affect the other and could thus lead to inconsistent behavior in the future.*

##### **Development problem 2:**
*There is a lack of documentation for the different surrogates and analytical models.*

To use the model instances for light curve prediction, we use their ``predict`` method. This method takes in a dictionary with the parameter names as keys (plus additionally ``luminosity_distance`` and ``redshift``) and then returns a time array (in days and observer frame) and a dictionary with the AB-magnitudes, having the photometric filters as keys.

In [ ]:
params_bu2026 = dict(
    redshift=0.009,
    luminosity_distance=40, # Mpc
    inclination_EM=0, # rad
    log10_mej_dyn=-2,
    v_ej_dyn=0.2,
    Ye_dyn=0.3,
    log10_mej_wind=-1.5,
    v_ej_wind=0.07,
    Ye_wind=0.35,    
)

times_bu2026, mags_bu2026 = bu2026_mlp.predict(params_bu2026)


params_metzger2017 = dict(
    redshift=0.009,
    luminosity_distance=40, # Mpc
    log10_mej=-1.38,
    log10_vej=-2.38,
    beta=1.2,
    log10_kappa_r=0,  
)

times_metzger2017, mags_metzger2017= metzger2017.predict(params_metzger2017)

We make a quick plot from the model's outputs

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(1,1, figsize=(8,5))

def plot_lcs(ax, times, mag_arr, color, linestyle="solid", label=None):
    ax.plot(times, mag_arr, color=color, linestyle=linestyle, label=label)
    ax.set_xscale("log")
    ax.set_ylabel("mag$_{\\mathrm{AB}}$")
    ax.set_xlabel("$t$ [days, observer frame]")
    ax.set_ylim(30, 15)

plot_lcs(ax, times_metzger2017, mags_metzger2017["besselli"], color="red", linestyle="dashed")
plot_lcs(ax, times_metzger2017, mags_metzger2017["bessellux"], color="blueviolet", linestyle="dashed")

plot_lcs(ax, times_bu2026, mags_bu2026["besselli"], label="$i$", color="red")
plot_lcs(ax, times_bu2026, mags_bu2026["bessellux"], label="$u$", color="blueviolet")


ax.legend(framealpha=1, fancybox=False)

fig.show()

For some models, e.g. the ``Bu2026_MLP`` surrogate here, we can even directly predict the entire flux density

In [ ]:
times, nus, log10_flux = bu2026_mlp.predict_log_flux(params_bu2026)

# make a 2D plot of the spectral flux density
fig = plt.figure(figsize=(6, 4))
gs = fig.add_gridspec(1, 2, width_ratios=[1, 0.05], wspace=0.35)

ax = fig.add_subplot(gs[0, 0])
cax = fig.add_subplot(gs[0, 1])

vmin = log10_flux[:, times>=0.8].min()
vmax = log10_flux[:, times>=0.8].max()

pcm0 = ax.pcolormesh(times, nus, log10_flux, cmap="inferno", vmin=vmin, vmax=vmax)
ax.set(xscale="log", yscale="log", xlim=(0.8, 25), ylim=(1e14, 2e15))
ax.set_ylabel("$\\nu$ [Hz, observer frame]")
ax.set_xlabel("$t$ [days, observer frame]")
ax.set_title("surrogate prediction", fontsize=15)

cbar = fig.colorbar(pcm0, cax=cax)

cbar.set_label(r'$\log_{10}(F_\nu)$ [mJy]')

Unless a particular model is trained directly on magnitudes or by design only predicts magnitudes (like SALT3), under the hood the magnitudes are always obtained by integrating the spectral range with the ``Filter`` classes:

In [ ]:
from fiesta.filters import Filter

bessellux_filter = Filter(name="bessellux") # here, it can be any name that is available in sncosmo

mag_u_arr = bessellux_filter.get_mag(10**log10_flux, nus)

You can even add a photometric filter after a model has been loaded.

In [ ]:
bu2026_mlp.add_filters(["bessellv", bessellux_filter, "X-ray-1keV"])
metzger2017.add_filters("bessellv")

You can also combine multiple models into a single model:

In [ ]:
import jax.numpy as jnp
from fiesta.inference.lightcurve_model import CombinedSurrogate

# load a model for GRB afterglow
afgpy_gaussian = FluxModel(
    "afgpy_gaussian_CVAE", 
    filters=["besselli", "radio-3GHz", "X-ray-1keV"]
)

# combine the Bu2026 model with GRB afterglow model 
combined_model = CombinedSurrogate(
    models = [bu2026_mlp, afgpy_gaussian],
    sample_times = jnp.geomspace(0.1, 50, 70)
)

##### **Exercise 1:**
*Get the $g$ band magnitude for the predicted light curve from the bu2026_MLP and metzger2017 model. Make a plot similar to the one above for the $i$ and $u$ band.*

In [ ]:
# your code



## **2. Training a surrogate**

Now we turn to the case where we want to train a new surrogate.

Specifically, we want to collectively train a surrogate for Daniel Brethauers non-LTE radiative transfer simulations with SEDONA (https://arxiv.org/pdf/2508.18364).

The training data can be downloaded from Zenodo (https://zenodo.org/records/22214049) and is roughly 1.4 GB.

<font color='red'>Warning: Only execute the following cell when you want to download 1.4 GB to the directory of this notebook. </font>

In [ ]:
!wget -O Brethauer_2026_training_data.zip https://zenodo.org/records/22214049/files/UpdatedQNLTEGrid.zip?download=1

##### **a) Creating a training data file**



In fiesta, training a machine-learning surrogate starts from a ``.h5`` file that contains all the raw training data plus the metadata.

This ``.h5`` file must have a certain layout. Specifically, it has to include the following data sets as metadata:

- ``times`` : An array for the time domain of the data in days.

- ``nus`` : An array for the frequency domain of the data in Hz.

- ``parameter_names`` : A list of strings that contains the parameter names. This determines which parameter names need to be present in the param-dict that is the argument for the surrogate prediction.

- ``parameter_distributions`` : A string-converted dictionary that has ``parameter_names`` as keys and the values are tuples ``tuple[float, float, str]``. The first two numbers are the minimum and maximum 
    range of this parameter in the training data, i.e., the range in which the trained surrogate will be valid. The string should indicate which distribution the training parameter samples follow, though there are no negative side-effects should the distribution provided here be inaccurate.

Further, there should be three data sets ``train``, ``val``, and ``test``. Each of these contains an array ``X`` and an array ``y``.

The ``X`` array is the array with the parameter vectors. The ``y`` array is the array with the flux densities at 10 pc (but zero redshift), in units of log10(mJy).

For fiesta it does not really matter, how this file is created, as long as it adheres to the aforementioned format. 
But of course, there are utility functions to convert the outputs of common codes like POSSIS or SEDONA to the fiesta format.

In [ ]:
from fiesta.train.utils import convert_SEDONA_outputs_to_h5


convert_SEDONA_outputs_to_h5(
    dirs="./Brethauer_2026_training_data",
    outfile="./Brethauer_2026_training_data.h5",
    parameter_names=["log10_X_lan", "vkin", "log10_mej"],
    log_arguments=[0, 2],
    clip=6.5144 # clips all entries to 0 abs. mag
)

##### **Exercise 2:**
*Open the fiesta training file that (hopefully) was just created. Look into the contents and familiarize yourself with the structure.*

In [ ]:
import h5py

with h5py.File("Brethauer_2026_training_data.h5", "r") as f:

    print(f.keys())

    print(f["times"][:])

    # your code here

##### **b) Training surrogates**

Now that we have a training data file, we can continue with our training procedure.

The intended way to interact with the training data file is through the ``DataManager`` class:

In [ ]:
from fiesta.train import DataManager


data = DataManager(
    file="Brethauer_2026_training_data.h5",
    tmin=0.1,
    tmax=20,
    numin=5e13,
    numax=1e15,
)

data.print_file_info()

Here, the ``tmin``, ``tmax``, ``numin``, and ``numax`` arguments are there to cut the flux densities to the desired time and frequency range.

We can now use the trainer classes to have a (more or less automatized) training process. Here we have it for the simple MLP architecture:

In [ ]:
from fiesta.train import PCATrainer, NeuralnetConfig
import jax


trainer_mlp = PCATrainer(
    model_name="Brethauer2026_MLP",
    outdir="Brethauer2026_MLP",
    plots_dir="Brethauer2026_MLP",
    data_manager=data,
    n_pca=15,
)

config = NeuralnetConfig(
    name="MLP",
    hidden_layer_sizes=[64, 64],
    learning_rate = 8e-3,

    batch_size=128,
    nb_epochs=1_500,

    weight_decay=0.0,
    dropout_rate=0.0,
    use_cosine_schedule=False,
    cosine_alpha=0.01,
    max_grad_norm=0.0,
)


trainer_mlp.fit(
    config=config,
    key=jax.random.key(47892),
    verbose=True
)

trainer_mlp.save()


trainer_mlp.plot_example_lc(["besselli", "bessellux"])


The other architecture implemented is the CVAE

In [ ]:
from fiesta.train import CVAETrainer
import numpy as np

trainer_cvae = CVAETrainer(
    model_name="Brethauer2026_CVAE",
    outdir="Brethauer2026_CVAE",
    data_manager=data,
    image_size=np.array([50, 75]),
)

config = NeuralnetConfig(
    name="CVAE",
    hidden_layer_sizes=[200, 75, 30],
    learning_rate = 3e-3,

    batch_size=256,
    nb_epochs=1_500,
    
    weight_decay=0.0
)

trainer_cvae.fit(
    config=config,
    key=jax.random.key(5781),
)


trainer_cvae.save()

trainer_cvae.plot_example_lc(["besselli", "bessellux"])

##### **Development problem 3:**
*The training API is not very flexible and there are only two different NN architectures.*

*There probably should be only one single trainer class for all architectures and we could try to unify / clean this up a little bit.*

*Also the interplay between ``DataManager`` and ``Trainer`` class is sometimes weird.*

##### **Development problem 4:**
*Very technical, but the pickling of the ImageScaler class can cause issues under certain circumstances.*

*Therefore, the ImageScaler should probably be refactored, but this would mean all CVAE models have to be retrained.*

##### **c) Benchmarking a surrogate**

Now, to check whether the training was actually successful, we need to do some benchmarking of our freshly made surrogates.
For this purpose, fiesta has a benchmarker functionality that will use the ``test`` data from the training data file.

In [ ]:
from fiesta.inference import FluxModel
from fiesta.train import Benchmarker

brethauer_2026_mlp = FluxModel("Brethauer2026_MLP", ["besselli", "ztfg", "bessellux"], directory="./Brethauer2026_MLP")
benchmarker_mlp = Benchmarker(
    model=brethauer_2026_mlp,
    data_file="./Brethauer_2026_training_data.h5",
    outdir="./Brethauer2026_MLP",
    metric_name="L2"
)

benchmarker_mlp.benchmark()

##### **Exercise 3:**
*Benchmark the CVAE model. Which architecture performs better?*


##### **Development problem 4:**
*The benchmarking API is not very up-to-date and probably we should overhaul it to include better benchmarks and metrics.*
*Also the documentation could be improved.*

##### **Exercise 4:**
*Tune the hyper-parameters for the MLP model. Who can find the best-performing setup?*
